# GPT IELTS Multi-Trait Scoring — Essay + Qwen-Style Discourse Features, 4 Trait Stacks

In [1]:
# =========================
# INSTALL + IMPORTS
# =========================

!pip -q install transformers

import os
import re
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Tuple
from collections import Counter

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import cohen_kappa_score, mean_absolute_error

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

try:
    from IPython.display import display
except Exception:
    display = print


In [2]:
# =========================
# REPRODUCIBILITY
# =========================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


In [3]:
# =========================
# CONFIG - GPT + QWEN-STYLE DISCOURSE FEATURES, 4 TRAIT STACKS
# =========================

OUTPUT_DIR = "/content/gpt_qwen_style_discourse_trait_stacks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = "/content/train.csv"
VAL_PATH   = "/content/val.csv"
TEST_PATH  = "/content/test.csv"

TRAIT_COLS = ["TR", "CC", "LR", "GRA"]

MIN_BAND = 0.0
MAX_BAND = 9.0

MODEL_NAME = "distilgpt2"
MAX_LEN = 512

STACK_HIDDEN = 512
DISCOURSE_HIDDEN = 128
DROPOUT = 0.2

BATCH_SIZE = 32
EPOCHS = 20
LR = 2e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.1

EARLY_STOPPING_PATIENCE = 4
MIN_DELTA = 1e-4

USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS = False
OVERALL_CONSISTENCY_WEIGHT = 0.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


DEVICE: cuda


In [4]:
# =========================
# LOAD PRE-SPLIT CSV FILES
# Upload train.csv, val.csv, test.csv to /content first
# =========================

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

display(train_df.head())

# =========================
# BASIC COLUMN CHECK
# =========================

required_cols = ["essay", "TR", "CC", "LR", "GRA"]

for name, df_ in [
    ("train_df", train_df),
    ("val_df", val_df),
    ("test_df", test_df),
]:
    missing = [c for c in required_cols if c not in df_.columns]

    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

    # If prompt is missing, create an empty prompt column.
    # The notebook still runs, but prompt-overlap features will be weak/zero.
    if "prompt" not in df_.columns:
        df_["prompt"] = ""

    df_["prompt"] = df_["prompt"].fillna("").astype(str)
    df_["essay"] = df_["essay"].fillna("").astype(str)

    for col in TRAIT_COLS:
        df_[col] = df_[col].astype(float)

    if "Overall" in df_.columns:
        df_["Overall"] = df_["Overall"].astype(float)

print("All datasets are ready.")

# =========================
# OVERLAP CHECK
# =========================

train_essays = set(train_df["essay"].astype(str))
val_essays = set(val_df["essay"].astype(str))
test_essays = set(test_df["essay"].astype(str))

print("Train-Val overlap:", len(train_essays & val_essays))
print("Train-Test overlap:", len(train_essays & test_essays))
print("Val-Test overlap:", len(val_essays & test_essays))

print("\nThis notebook does NOT auto-detect numeric CSV columns as discourse features.")
print("Discourse features are extracted from prompt + essay only.")


Train: (7743, 17)
Val: (969, 17)
Test: (970, 17)

Train columns:
['prompt', 'essay', 'evaluation', 'essay_len', 'TR', 'CC', 'LR', 'GRA', 'n_found', 'parse_quality', 'overall_raw', 'band', 'prompt_relevance', 'lexical_diversity', 'readability_score', 'target_text', 'source']


,prompt,essay,evaluation,essay_len,TR,CC,LR,GRA,n_found,parse_quality,overall_raw,band,prompt_relevance,lexical_diversity,readability_score,target_text,source
0,Some employers believe that job applicants’ so...,There has been much discussion revolving aroun...,Task Achievement:\r\n\r\n- The candidate has e...,273,7.5,7.0,7.0,7.0,4,good,7.125,7.0,0.592603,0.581818,40.754804,Analysis: This essay has a lexical diversity o...,hf
1,Some people believe that teenagers should be r...,A nation is known as a vast garden and childre...,Task Achievement:\r\nThe essay effectively add...,324,4.0,4.0,3.0,3.0,4,good,3.500,3.5,0.600165,0.563077,35.677525,Analysis: This essay has a lexical diversity o...,hf
2,some people say that economic growth is the on...,"In the modern world ,a burning issue arises th...",Task Achievement:\r\n\r\nThe essay adequately ...,348,5.5,5.0,5.0,5.0,4,good,5.125,5.0,0.746589,0.452450,40.414844,Analysis: This essay has a lexical diversity o...,hf
3,Some believe that people should make efforts t...,The issue of climate change was always debatab...,Task Achievement:\r\n- The candidate has adequ...,307,7.0,7.0,6.0,6.0,4,good,6.500,6.5,0.628859,0.629870,39.378580,Analysis: This essay has a lexical diversity o...,hf
4,Nowadays families move to different countries ...,Immigrating to other nations for business purp...,Task Achievement:\r\nThe candidate has effecti...,311,7.5,8.0,7.0,7.5,4,good,7.500,7.5,0.586910,0.612179,40.385892,Analysis: This essay has a lexical diversity o...,hf


All datasets are ready.
Train-Val overlap: 0
Train-Test overlap: 1
Val-Test overlap: 0

This notebook does NOT auto-detect numeric CSV columns as discourse features.
Discourse features are extracted from prompt + essay only.


In [5]:
# =========================
# METRIC FUNCTIONS
# Includes derived Overall QWK
# =========================

def band_to_scaled(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - MIN_BAND) / (MAX_BAND - MIN_BAND)


def scaled_to_band(x):
    x = np.asarray(x, dtype=np.float32)

    band = x * (MAX_BAND - MIN_BAND) + MIN_BAND
    band = np.clip(band, MIN_BAND, MAX_BAND)

    # IELTS half-band rounding
    band = np.round(band * 2) / 2

    return band


def qwk_band(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    valid = ~np.isnan(y_true) & ~np.isnan(y_pred)

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return np.nan

    y_true_int = (y_true * 2).astype(int)
    y_pred_int = (y_pred * 2).astype(int)

    return cohen_kappa_score(
        y_true_int,
        y_pred_int,
        weights="quadratic",
    )


def compute_metrics(y_true_scaled, y_pred_scaled, overall_true=None):
    y_true_band = scaled_to_band(y_true_scaled)
    y_pred_band = scaled_to_band(y_pred_scaled)

    metrics = {}

    trait_qwks = []
    trait_maes = []

    for i, trait in enumerate(TRAIT_COLS):
        qwk = qwk_band(y_true_band[:, i], y_pred_band[:, i])
        mae = mean_absolute_error(y_true_band[:, i], y_pred_band[:, i])

        metrics[f"{trait}_qwk"] = qwk
        metrics[f"{trait}_mae"] = mae

        trait_qwks.append(qwk)
        trait_maes.append(mae)

    metrics["mean_trait_qwk"] = float(np.nanmean(trait_qwks))
    metrics["mean_trait_mae"] = float(np.nanmean(trait_maes))

    # Derived overall = average of 4 predicted trait bands
    derived_overall_pred = np.mean(y_pred_band, axis=1)
    derived_overall_pred = np.round(derived_overall_pred * 2) / 2
    derived_overall_pred = np.clip(derived_overall_pred, MIN_BAND, MAX_BAND)

    # Derived overall compared to derived gold overall from gold traits
    derived_overall_gold_from_traits = np.mean(y_true_band, axis=1)
    derived_overall_gold_from_traits = np.round(derived_overall_gold_from_traits * 2) / 2
    derived_overall_gold_from_traits = np.clip(derived_overall_gold_from_traits, MIN_BAND, MAX_BAND)

    metrics["derived_overall_from_traits_qwk"] = qwk_band(
        derived_overall_gold_from_traits,
        derived_overall_pred,
    )

    metrics["derived_overall_from_traits_mae"] = mean_absolute_error(
        derived_overall_gold_from_traits,
        derived_overall_pred,
    )

    # Derived overall compared to real Overall column, if available
    if overall_true is not None:
        overall_true = np.asarray(overall_true, dtype=np.float32)
        valid = ~np.isnan(overall_true)

        if valid.any():
            metrics["derived_overall_qwk"] = qwk_band(
                overall_true[valid],
                derived_overall_pred[valid],
            )

            metrics["derived_overall_mae"] = mean_absolute_error(
                overall_true[valid],
                derived_overall_pred[valid],
            )

    return metrics, y_true_band, y_pred_band, derived_overall_pred


In [6]:
# =========================
# QWEN-STYLE DISCOURSE FEATURE EXTRACTOR
# Extracts features from prompt + essay only
# =========================

STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "while", "of", "to", "in", "on", "for",
    "with", "as", "by", "at", "from", "up", "down", "out", "over", "under", "again",
    "further", "then", "once", "here", "there", "when", "where", "why", "how", "all",
    "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor",
    "not", "only", "own", "same", "so", "than", "too", "very", "can", "will", "just",
    "should", "now", "is", "am", "are", "was", "were", "be", "been", "being", "do",
    "does", "did", "have", "has", "had", "i", "you", "he", "she", "it", "we", "they",
    "me", "him", "her", "us", "them", "my", "your", "his", "its", "our", "their",
}

CONNECTIVE_CATEGORIES = {
    "additive": {
        "also", "moreover", "furthermore", "in addition", "besides", "additionally",
        "not only", "as well as",
    },
    "contrastive": {
        "however", "nevertheless", "nonetheless", "although", "though", "even though",
        "whereas", "while", "on the other hand", "in contrast", "by contrast", "but",
    },
    "causal": {
        "because", "since", "as a result", "therefore", "thus", "hence", "consequently",
        "so", "due to", "owing to",
    },
    "temporal": {
        "first", "second", "third", "finally", "then", "next", "after", "before",
        "meanwhile", "subsequently",
    },
    "exemplification": {
        "for example", "for instance", "such as", "namely", "including",
    },
    "conclusive": {
        "in conclusion", "to conclude", "overall", "in summary", "to sum up",
    },
}

SUBORDINATORS = {
    "although", "because", "since", "while", "whereas", "if", "unless", "when", "after",
    "before", "even though", "as soon as", "provided that",
}

ACADEMIC_WORDS = {
    "benefit", "beneficial", "significant", "consequence", "issue", "factor", "approach",
    "impact", "affect", "society", "individual", "environment", "development", "evidence",
    "argument", "perspective", "policy", "economic", "education", "technology", "global",
    "essential", "necessary", "advantage", "disadvantage", "solution", "challenge",
}

THESIS_MARKERS = {
    "i believe", "i think", "in my opinion", "this essay", "i agree", "i disagree",
    "it is argued", "it can be argued",
}

POSITION_MARKERS = {
    "agree", "disagree", "support", "oppose", "believe", "argue", "claim", "view",
}


def simple_word_tokenize(text):
    text = str(text).lower()
    return re.findall(r"[a-z]+(?:'[a-z]+)?|[0-9]+", text)


def simple_sentence_split(text):
    text = str(text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences if len(sentences) > 0 else [text] if text else []


def simple_paragraph_split(text):
    paragraphs = re.split(r"\n\s*\n|\r\n\s*\r\n", str(text).strip())
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    return paragraphs if len(paragraphs) > 0 else [str(text).strip()] if str(text).strip() else []


def safe_div(a, b):
    return float(a) / float(b) if b else 0.0


def content_words(tokens):
    return [t for t in tokens if t not in STOPWORDS and len(t) > 2]


def rough_stem(word):
    for suffix in ["ing", "edly", "edly", "ed", "ly", "es", "s", "ment", "tion", "ions"]:
        if word.endswith(suffix) and len(word) > len(suffix) + 3:
            return word[: -len(suffix)]
    return word


def ngrams(tokens, n):
    if len(tokens) < n:
        return []
    return list(zip(*[tokens[i:] for i in range(n)]))


def count_phrase_set(text, phrases):
    text_l = str(text).lower()
    count = 0
    for p in phrases:
        if " " in p:
            count += text_l.count(p)
        else:
            count += len(re.findall(rf"\b{re.escape(p)}\b", text_l))
    return count


def count_connective_categories(text):
    total = 0
    counts = {}
    for cat, phrases in CONNECTIVE_CATEGORIES.items():
        c = count_phrase_set(text, phrases)
        counts[f"{cat}_connective_count"] = c
        total += c
    counts["connective_count"] = total
    return counts


def adjacent_sentence_overlap(sentences):
    overlaps = []
    for s1, s2 in zip(sentences[:-1], sentences[1:]):
        w1 = set(content_words(simple_word_tokenize(s1)))
        w2 = set(content_words(simple_word_tokenize(s2)))
        overlaps.append(safe_div(len(w1 & w2), len(w1 | w2)))
    if not overlaps:
        return {
            "adjacent_sentence_overlap_mean": 0.0,
            "adjacent_sentence_overlap_std": 0.0,
            "adjacent_sentence_overlap_max": 0.0,
        }
    return {
        "adjacent_sentence_overlap_mean": float(np.mean(overlaps)),
        "adjacent_sentence_overlap_std": float(np.std(overlaps)),
        "adjacent_sentence_overlap_max": float(np.max(overlaps)),
    }


def paragraph_balance(paragraphs):
    lens = [len(simple_word_tokenize(p)) for p in paragraphs]
    if len(lens) <= 1:
        return 1.0
    mean_len = np.mean(lens)
    return float(1.0 - min(1.0, safe_div(np.std(lens), mean_len + 1e-6)))


def lexical_chain_proxy(tokens):
    cwords = [rough_stem(w) for w in content_words(tokens)]
    counts = Counter(cwords)
    chains = [c for c in counts.values() if c >= 2]

    large_chains = [c for c in chains if c >= 3]

    return {
        "lexical_chain_count": len(chains),
        "avg_lexical_chain_size": float(np.mean(chains)) if chains else 0.0,
        "num_large_lexical_chains": len(large_chains),
        "num_large_varied_lexical_chains": len(large_chains),
        "pct_large_lexical_chains": safe_div(len(large_chains), len(chains)),
        "pct_large_varied_lexical_chains": safe_div(len(large_chains), len(chains)),
        "lexical_chain_coverage": safe_div(sum(chains), len(cwords)),
        "lexical_chain_variety": safe_div(len(chains), len(set(cwords))),
    }


def syllable_proxy(word):
    word = word.lower()
    groups = re.findall(r"[aeiouy]+", word)
    return max(1, len(groups))


def readability_features(tokens, sentences):
    word_count = len(tokens)
    sentence_count = len(sentences)
    syllables = sum(syllable_proxy(w) for w in tokens)

    syllable_per_word = safe_div(syllables, word_count)
    words_per_sentence = safe_div(word_count, sentence_count)

    flesch = 206.835 - 1.015 * words_per_sentence - 84.6 * syllable_per_word
    fk_grade = 0.39 * words_per_sentence + 11.8 * syllable_per_word - 15.59

    return {
        "syllable_proxy_per_word": syllable_per_word,
        "flesch_reading_ease_proxy": flesch,
        "flesch_kincaid_grade_proxy": fk_grade,
    }


def extract_discourse_features(prompt, essay):
    prompt = str(prompt)
    essay = str(essay)

    tokens = simple_word_tokenize(essay)
    prompt_tokens = simple_word_tokenize(prompt)

    sentences = simple_sentence_split(essay)
    paragraphs = simple_paragraph_split(essay)

    word_count = len(tokens)
    sentence_count = len(sentences)
    paragraph_count = len(paragraphs)

    unique_words = set(tokens)

    sent_lens = [len(simple_word_tokenize(s)) for s in sentences]
    avg_sentence_len = float(np.mean(sent_lens)) if sent_lens else 0.0
    sentence_len_std = float(np.std(sent_lens)) if sent_lens else 0.0

    prompt_content = set(content_words(prompt_tokens))
    essay_content = set(content_words(tokens))

    prompt_overlap = safe_div(len(prompt_content & essay_content), len(prompt_content))

    prompt_bigrams = set(ngrams(content_words(prompt_tokens), 2))
    essay_bigrams = set(ngrams(content_words(tokens), 2))

    prompt_bigram_overlap = safe_div(len(prompt_bigrams & essay_bigrams), len(prompt_bigrams))

    text_l = essay.lower()

    connective_counts = count_connective_categories(essay)
    overlap_feats = adjacent_sentence_overlap(sentences)
    chain_feats = lexical_chain_proxy(tokens)
    read_feats = readability_features(tokens, sentences)

    bigram_count = len(ngrams(tokens, 2))
    trigram_count = len(ngrams(tokens, 3))

    bigram_diversity = safe_div(len(set(ngrams(tokens, 2))), bigram_count)
    trigram_diversity = safe_div(len(set(ngrams(tokens, 3))), trigram_count)

    punctuation_count = len(re.findall(r"[.,;:!?]", essay))
    comma_count = essay.count(",")
    semicolon_colon_count = essay.count(";") + essay.count(":")
    question_exclaim_count = essay.count("?") + essay.count("!")

    subordinators = count_phrase_set(essay, SUBORDINATORS)

    very_long_sentences = sum(1 for l in sent_lens if l >= 35)
    very_short_sentences = sum(1 for l in sent_lens if l <= 5)

    complex_sentences = 0
    for s in sentences:
        if count_phrase_set(s, SUBORDINATORS) > 0 or "," in s:
            complex_sentences += 1

    first_person = {"i", "me", "my", "mine", "we", "us", "our", "ours"}
    modals = {"can", "could", "may", "might", "must", "should", "would", "will"}

    feats = {
        "word_count": word_count,
        "sentence_count": sentence_count,
        "paragraph_count": paragraph_count,
        "prompt_word_count": len(prompt_tokens),
        "prompt_overlap": prompt_overlap,
        "prompt_bigram_overlap": prompt_bigram_overlap,

        "avg_sentence_len": avg_sentence_len,
        "sentence_len_std": sentence_len_std,
        "paragraph_balance": paragraph_balance(paragraphs),

        "unique_word_count": len(unique_words),
        "avg_word_len": float(np.mean([len(w) for w in tokens])) if tokens else 0.0,
        "type_token_ratio": safe_div(len(unique_words), word_count),
        "root_ttr": safe_div(len(unique_words), math.sqrt(word_count)),
        "hapax_ratio": safe_div(sum(1 for c in Counter(tokens).values() if c == 1), word_count),
        "long_word_ratio": safe_div(sum(1 for w in tokens if len(w) >= 7), word_count),
        "bigram_diversity": bigram_diversity,
        "trigram_diversity": trigram_diversity,
        "academic_word_ratio": safe_div(sum(1 for w in tokens if w in ACADEMIC_WORDS), word_count),

        "modal_ratio": safe_div(sum(1 for w in tokens if w in modals), word_count),
        "first_person_ratio": safe_div(sum(1 for w in tokens if w in first_person), word_count),
        "thesis_marker_count": count_phrase_set(essay, THESIS_MARKERS),
        "position_marker_count": count_phrase_set(essay, POSITION_MARKERS),

        "comma_per_sentence": safe_div(comma_count, sentence_count),
        "semicolon_colon_per_sentence": safe_div(semicolon_colon_count, sentence_count),
        "question_exclaim_per_sentence": safe_div(question_exclaim_count, sentence_count),
        "punctuation_error_proxy": safe_div(max(0, punctuation_count - sentence_count * 4), sentence_count),

        "subordinator_count": subordinators,
        "subordinator_ratio": safe_div(subordinators, sentence_count),
        "complex_sentence_ratio": safe_div(complex_sentences, sentence_count),
        "very_long_sentence_ratio": safe_div(very_long_sentences, sentence_count),
        "very_short_sentence_ratio": safe_div(very_short_sentences, sentence_count),
    }

    feats.update(connective_counts)
    feats["connective_per_sentence"] = safe_div(feats["connective_count"], sentence_count)

    feats.update(overlap_feats)
    feats.update(chain_feats)
    feats.update(read_feats)

    return feats


FEATURE_GROUPS = {
    "TR": [
        "word_count",
        "sentence_count",
        "prompt_word_count",
        "prompt_overlap",
        "prompt_bigram_overlap",
        "modal_ratio",
        "first_person_ratio",
        "thesis_marker_count",
        "position_marker_count",
    ],

    "CC": [
        "paragraph_count",
        "sentence_count",
        "avg_sentence_len",
        "sentence_len_std",
        "adjacent_sentence_overlap_mean",
        "adjacent_sentence_overlap_std",
        "adjacent_sentence_overlap_max",
        "paragraph_balance",
        "connective_count",
        "connective_per_sentence",
        "additive_connective_count",
        "contrastive_connective_count",
        "causal_connective_count",
        "temporal_connective_count",
        "exemplification_connective_count",
        "conclusive_connective_count",
        "lexical_chain_count",
        "avg_lexical_chain_size",
        "num_large_lexical_chains",
        "num_large_varied_lexical_chains",
        "pct_large_lexical_chains",
        "pct_large_varied_lexical_chains",
        "lexical_chain_coverage",
    ],

    "LR": [
        "word_count",
        "unique_word_count",
        "avg_word_len",
        "type_token_ratio",
        "root_ttr",
        "hapax_ratio",
        "long_word_ratio",
        "bigram_diversity",
        "trigram_diversity",
        "academic_word_ratio",
        "lexical_chain_count",
        "lexical_chain_variety",
        "lexical_chain_coverage",
    ],

    "GRA": [
        "sentence_count",
        "avg_sentence_len",
        "sentence_len_std",
        "comma_per_sentence",
        "semicolon_colon_per_sentence",
        "subordinator_count",
        "subordinator_ratio",
        "complex_sentence_ratio",
        "very_long_sentence_ratio",
        "very_short_sentence_ratio",
        "punctuation_error_proxy",
        "question_exclaim_per_sentence",
        "syllable_proxy_per_word",
        "flesch_reading_ease_proxy",
        "flesch_kincaid_grade_proxy",
    ],
}

ALL_FEATURES = sorted(set(sum(FEATURE_GROUPS.values(), [])))

FEATURE_IDXS = {
    trait: [ALL_FEATURES.index(f) for f in feats]
    for trait, feats in FEATURE_GROUPS.items()
}

FEATURE_DIMS = {
    trait: len(FEATURE_IDXS[trait])
    for trait in TRAIT_COLS
}

print("Total discourse features:", len(ALL_FEATURES))
for trait in TRAIT_COLS:
    print(trait, FEATURE_DIMS[trait], [ALL_FEATURES[i] for i in FEATURE_IDXS[trait]])


class FeatureNormalizer:
    def __init__(self, feature_names):
        self.feature_names = feature_names
        self.mean = None
        self.std = None

    def fit(self, df):
        feats = []

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Fit discourse normalizer"):
            d = extract_discourse_features(row["prompt"], row["essay"])
            feats.append([d.get(f, 0.0) for f in self.feature_names])

        arr = np.asarray(feats, dtype=np.float32)

        self.mean = arr.mean(axis=0)
        self.std = arr.std(axis=0)
        self.std[self.std < 1e-6] = 1.0

        return self

    def transform_row(self, prompt, essay):
        d = extract_discourse_features(prompt, essay)

        arr = np.asarray(
            [d.get(f, 0.0) for f in self.feature_names],
            dtype=np.float32,
        )

        return (arr - self.mean) / self.std


feature_normalizer = FeatureNormalizer(ALL_FEATURES).fit(train_df)

DISC_FEAT_DIM = len(ALL_FEATURES)

example_features = feature_normalizer.transform_row(
    train_df.loc[0, "prompt"],
    train_df.loc[0, "essay"],
)

print("DISC_FEAT_DIM:", DISC_FEAT_DIM)
print("Example feature vector shape:", example_features.shape)
print("First 5 normalized features:", example_features[:5])


Total discourse features: 53
TR 9 ['word_count', 'sentence_count', 'prompt_word_count', 'prompt_overlap', 'prompt_bigram_overlap', 'modal_ratio', 'first_person_ratio', 'thesis_marker_count', 'position_marker_count']
CC 23 ['paragraph_count', 'sentence_count', 'avg_sentence_len', 'sentence_len_std', 'adjacent_sentence_overlap_mean', 'adjacent_sentence_overlap_std', 'adjacent_sentence_overlap_max', 'paragraph_balance', 'connective_count', 'connective_per_sentence', 'additive_connective_count', 'contrastive_connective_count', 'causal_connective_count', 'temporal_connective_count', 'exemplification_connective_count', 'conclusive_connective_count', 'lexical_chain_count', 'avg_lexical_chain_size', 'num_large_lexical_chains', 'num_large_varied_lexical_chains', 'pct_large_lexical_chains', 'pct_large_varied_lexical_chains', 'lexical_chain_coverage']
LR 13 ['word_count', 'unique_word_count', 'avg_word_len', 'type_token_ratio', 'root_ttr', 'hapax_ratio', 'long_word_ratio', 'bigram_diversity', 'tr

Fit discourse normalizer:   0%|          | 0/7743 [00:00<?, ?it/s]

DISC_FEAT_DIM: 53
Example feature vector shape: (53,)
First 5 normalized features: [-0.59143883 -1.2670257  -0.84109366 -0.6860358  -0.73181754]


In [7]:
# =========================
# TOKENIZER
# GPT tokenizers usually do not have a pad token, so we use eos_token as pad_token.
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded tokenizer:", MODEL_NAME)
print("Pad token:", tokenizer.pad_token, tokenizer.pad_token_id)

print("Loaded tokenizer:", MODEL_NAME)

# =========================
# DATASET + COLLATOR - ESSAY + QWEN-STYLE DISCOURSE FEATURES
# =========================

class IELTSGPTDiscourseDataset(Dataset):
    def __init__(self, df, tokenizer, feature_normalizer, max_len=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.feature_normalizer = feature_normalizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoded = self.tokenizer(
            str(row["essay"]),
            truncation=True,
            padding=False,
            max_length=self.max_len,
        )

        discourse_features = self.feature_normalizer.transform_row(
            row["prompt"],
            row["essay"],
        ).astype(np.float32)

        labels_band = row[TRAIT_COLS].astype(float).values.astype(np.float32)
        labels_scaled = band_to_scaled(labels_band).astype(np.float32)

        if "Overall" in self.df.columns and not pd.isna(row.get("Overall", np.nan)):
            overall_band = np.float32(row["Overall"])
        else:
            overall_band = np.float32(np.nan)

        return {
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
            "discourse_features": discourse_features,
            "labels": labels_scaled,
            "overall_band": overall_band,
        }


@dataclass
class GPTDiscourseDataCollator:
    tokenizer: object

    def __call__(self, batch):
        features = [
            {
                "input_ids": x["input_ids"],
                "attention_mask": x["attention_mask"],
            }
            for x in batch
        ]

        padded = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt",
        )

        padded["discourse_features"] = torch.tensor(
            np.stack([x["discourse_features"] for x in batch]),
            dtype=torch.float32,
        )

        padded["labels"] = torch.tensor(
            np.stack([x["labels"] for x in batch]),
            dtype=torch.float32,
        )

        padded["overall_band"] = torch.tensor(
            [x["overall_band"] for x in batch],
            dtype=torch.float32,
        )

        return padded


train_dataset = IELTSGPTDiscourseDataset(train_df, tokenizer, feature_normalizer, max_len=MAX_LEN)
val_dataset   = IELTSGPTDiscourseDataset(val_df, tokenizer, feature_normalizer, max_len=MAX_LEN)
test_dataset  = IELTSGPTDiscourseDataset(test_df, tokenizer, feature_normalizer, max_len=MAX_LEN)

collator = GPTDiscourseDataCollator(tokenizer=tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded tokenizer: distilgpt2
Pad token: <|endoftext|> 50256
Loaded tokenizer: distilgpt2


In [8]:
# =========================
# MODEL - GPT + QWEN-STYLE DISCOURSE FEATURES + 4 TRAIT STACKS
# =========================

class TraitStack(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),

            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 4, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class GPTDiscourseTraitScorer(nn.Module):
    def __init__(
        self,
        model_name,
        feature_dims,
        feature_idxs,
        discourse_hidden=128,
        stack_hidden=512,
        dropout=0.2,
        pad_token_id=None,
    ):
        super().__init__()

        self.feature_idxs = feature_idxs

        self.gpt = AutoModel.from_pretrained(model_name)

        if pad_token_id is not None:
            self.gpt.config.pad_token_id = pad_token_id

        gpt_hidden = self.gpt.config.hidden_size

        self.gpt_norm = nn.LayerNorm(gpt_hidden)
        self.gpt_dropout = nn.Dropout(dropout)

        self.feature_projectors = nn.ModuleDict({
            trait: nn.Sequential(
                nn.LayerNorm(feature_dims[trait]),
                nn.Linear(feature_dims[trait], discourse_hidden),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            for trait in TRAIT_COLS
        })

        fusion_dim = gpt_hidden + discourse_hidden

        self.stacks = nn.ModuleDict({
            trait: TraitStack(
                input_dim=fusion_dim,
                hidden_dim=stack_hidden,
                dropout=dropout,
            )
            for trait in TRAIT_COLS
        })

    def last_token_pooling(self, last_hidden_state, attention_mask):
        # Use the last non-padding token, which is natural for decoder-only GPT models.
        lengths = attention_mask.sum(dim=1) - 1
        batch_idx = torch.arange(last_hidden_state.size(0), device=last_hidden_state.device)
        return last_hidden_state[batch_idx, lengths]

    def forward(self, input_ids, attention_mask, discourse_features):
        outputs = self.gpt(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        essay_repr = self.last_token_pooling(outputs.last_hidden_state, attention_mask)
        essay_repr = self.gpt_norm(essay_repr)
        essay_repr = self.gpt_dropout(essay_repr)

        preds = []

        for trait in TRAIT_COLS:
            idxs = torch.tensor(
                self.feature_idxs[trait],
                dtype=torch.long,
                device=discourse_features.device,
            )

            trait_features = discourse_features.index_select(dim=1, index=idxs)
            trait_feature_repr = self.feature_projectors[trait](trait_features)

            trait_repr = torch.cat([essay_repr, trait_feature_repr], dim=-1)
            pred = self.stacks[trait](trait_repr)

            preds.append(pred)

        preds = torch.stack(preds, dim=1)
        preds = torch.sigmoid(preds)

        return preds


model = GPTDiscourseTraitScorer(
    model_name=MODEL_NAME,
    feature_dims=FEATURE_DIMS,
    feature_idxs=FEATURE_IDXS,
    discourse_hidden=DISCOURSE_HIDDEN,
    stack_hidden=STACK_HIDDEN,
    dropout=DROPOUT,
    pad_token_id=tokenizer.pad_token_id,
).to(DEVICE)

print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPTDiscourseTraitScorer(
  (gpt): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (gpt_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (gpt_dropout): Dropout(p=0.2, inplace=False)
  (feature_

In [9]:
# =========================
# LOSS
# =========================

def criterion_loss(pred_scaled, labels_scaled, overall_band=None):
    loss = 0.0

    for i, trait in enumerate(TRAIT_COLS):
        loss = loss + F.mse_loss(pred_scaled[:, i], labels_scaled[:, i])

    loss = loss / len(TRAIT_COLS)

    if USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS and overall_band is not None:
        valid = ~torch.isnan(overall_band)

        if valid.any():
            pred_overall_scaled = pred_scaled.mean(dim=1)

            gold_overall_scaled = (
                overall_band.to(pred_scaled.device) - MIN_BAND
            ) / (MAX_BAND - MIN_BAND)

            loss = loss + OVERALL_CONSISTENCY_WEIGHT * F.mse_loss(
                pred_overall_scaled[valid],
                gold_overall_scaled[valid],
            )

    return loss


In [10]:
# =========================
# EVALUATION HELPERS - GPT + DISCOURSE
# =========================

def move_batch_to_device(batch):
    out = {}

    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(DEVICE)
        else:
            out[k] = v

    return out


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()

    all_preds = []
    all_labels = []
    all_overall = []

    for batch in tqdm(loader, desc="Predict"):
        batch = move_batch_to_device(batch)

        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            discourse_features=batch["discourse_features"],
        )

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())
        all_overall.append(batch["overall_band"].detach().cpu().numpy())

    return (
        np.concatenate(all_labels, axis=0),
        np.concatenate(all_preds, axis=0),
        np.concatenate(all_overall, axis=0),
    )


def print_metrics(metrics, title):
    print("\n" + title)
    print("=" * len(title))

    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")


def evaluate_model(model, loader, name="val"):
    y_true, y_pred, overall_true = predict_loader(model, loader)

    metrics, y_true_band, y_pred_band, overall_pred = compute_metrics(
        y_true,
        y_pred,
        overall_true,
    )

    print_metrics(metrics, f"== {name} ==")

    return metrics, y_true_band, y_pred_band, overall_pred


In [11]:
# =========================
# TRAIN GPT + QWEN-STYLE DISCOURSE MODEL WITH EARLY STOPPING
# Main metric: mean_trait_qwk
# =========================

# Use smaller LR for GPT, larger LR for new task-specific heads
gpt_params = []
head_params = []

for name, param in model.named_parameters():
    if name.startswith("gpt."):
        gpt_params.append(param)
    else:
        head_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": gpt_params, "lr": LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

total_training_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

epochs_no_improve = 0
best_val_qwk = -1.0

best_path = os.path.join(OUTPUT_DIR, "best_gpt_qwen_style_discourse.pt")

for epoch in range(1, EPOCHS + 1):
    model.train()

    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for step, batch in enumerate(pbar, start=1):
        batch = move_batch_to_device(batch)

        optimizer.zero_grad(set_to_none=True)

        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            discourse_features=batch["discourse_features"],
        )

        loss = criterion_loss(
            preds,
            batch["labels"],
            batch["overall_band"],
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM,
        )

        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        avg_running_loss = running_loss / step

        pbar.set_postfix({
            "loss": f"{avg_running_loss:.6f}",
            "gpt_lr": optimizer.param_groups[0]["lr"],
            "head_lr": optimizer.param_groups[1]["lr"],
        })

    avg_train_loss = running_loss / len(train_loader)

    print(f"\nEpoch {epoch} train loss: {avg_train_loss:.6f}")

    val_metrics, _, _, _ = evaluate_model(
        model,
        val_loader,
        name=f"val epoch {epoch}",
    )

    current_qwk = val_metrics["mean_trait_qwk"]
    selected_metric_name = "mean_trait_qwk"

    print(f"Selected validation metric: {selected_metric_name}")
    print(f"Current val QWK: {current_qwk:.4f}")
    print(f"Best val QWK so far: {best_val_qwk:.4f}")

    if "derived_overall_qwk" in val_metrics:
        print(f"Val derived_overall_qwk: {val_metrics['derived_overall_qwk']:.4f}")

    improved = current_qwk > best_val_qwk + MIN_DELTA

    if improved:
        best_val_qwk = current_qwk
        epochs_no_improve = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_qwk": best_val_qwk,
                "selected_metric_name": selected_metric_name,
                "best_val_metrics": val_metrics,
                "feature_mean": feature_normalizer.mean.tolist(),
                "feature_std": feature_normalizer.std.tolist(),
                "config": {
                    "FEATURE_MODE": "gpt_qwen_style_discourse",
                    "MODEL_NAME": MODEL_NAME,
                    "MAX_LEN": MAX_LEN,
                    "STACK_HIDDEN": STACK_HIDDEN,
                    "DISCOURSE_HIDDEN": DISCOURSE_HIDDEN,
                    "DROPOUT": DROPOUT,
                    "BATCH_SIZE": BATCH_SIZE,
                    "EPOCHS": EPOCHS,
                    "LR": LR,
                    "HEAD_LR": HEAD_LR,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "MAX_GRAD_NORM": MAX_GRAD_NORM,
                    "TRAIT_COLS": TRAIT_COLS,
                    "MIN_BAND": MIN_BAND,
                    "MAX_BAND": MAX_BAND,
                    "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
                    "MIN_DELTA": MIN_DELTA,
                    "ALL_FEATURES": ALL_FEATURES,
                    "FEATURE_GROUPS": FEATURE_GROUPS,
                    "FEATURE_IDXS": FEATURE_IDXS,
                    "FEATURE_DIMS": FEATURE_DIMS,
                },
            },
            best_path,
        )

        print(f"[SAVE] Best model saved to: {best_path}")
        print(f"[SAVE] best_val_qwk = {best_val_qwk:.4f}")

    else:
        epochs_no_improve += 1

        print(
            f"[EARLY STOPPING] No improvement for "
            f"{epochs_no_improve}/{EARLY_STOPPING_PATIENCE} epochs."
        )

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print("[EARLY STOPPING] Triggered.")
            print(f"Stopped at epoch {epoch}.")
            break

    print("-" * 80)

print("\nTraining finished.")
print(f"Best validation QWK: {best_val_qwk:.4f}")
print(f"Best checkpoint path: {best_path}")


Epoch 1/20:   0%|          | 0/242 [00:00<?, ?it/s]


Epoch 1 train loss: 0.040162


Predict:   0%|          | 0/31 [00:00<?, ?it/s]


== val epoch 1 ==
TR_qwk: 0.1850
TR_mae: 1.0470
CC_qwk: 0.2017
CC_mae: 1.1698
LR_qwk: 0.2326
LR_mae: 1.0831
GRA_qwk: 0.2380
GRA_mae: 1.1496
mean_trait_qwk: 0.2143
mean_trait_mae: 1.1124
derived_overall_from_traits_qwk: 0.1990
derived_overall_from_traits_mae: 1.1032
Selected validation metric: mean_trait_qwk
Current val QWK: 0.2143
Best val QWK so far: -1.0000
[SAVE] Best model saved to: /content/gpt_qwen_style_discourse_trait_stacks/best_gpt_qwen_style_discourse.pt
[SAVE] best_val_qwk = 0.2143
--------------------------------------------------------------------------------


Epoch 2/20:   0%|          | 0/242 [00:00<?, ?it/s]

...


== val epoch 13 ==
TR_qwk: 0.6321
TR_mae: 0.9133
CC_qwk: 0.6245
CC_mae: 1.0253
LR_qwk: 0.6691
LR_mae: 0.9303
GRA_qwk: 0.6758
GRA_mae: 0.9458
mean_trait_qwk: 0.6504
mean_trait_mae: 0.9537
derived_overall_from_traits_qwk: 0.6562
derived_overall_from_traits_mae: 0.9319
Selected validation metric: mean_trait_qwk
Current val QWK: 0.6504
Best val QWK so far: 0.6581
[EARLY STOPPING] No improvement for 3/4 epochs.
--------------------------------------------------------------------------------


Epoch 14/20:   0%|          | 0/242 [00:00<?, ?it/s]


Epoch 14 train loss: 0.008127


Predict:   0%|          | 0/31 [00:00<?, ?it/s]


== val epoch 14 ==
TR_qwk: 0.5778
TR_mae: 0.9567
CC_qwk: 0.5716
CC_mae: 1.0722
LR_qwk: 0.6283
LR_mae: 0.9350
GRA_qwk: 0.6379
GRA_mae: 0.9649
mean_trait_qwk: 0.6039
mean_trait_mae: 0.9822
derived_overall_from_traits_qwk: 0.6098
derived_overall_from_traits_mae: 0.9530
Selected validation metric: mean_trait_qwk
Current val QWK: 0.6039
Best val QWK so far: 0.6581
[EARLY STOPPING] No improvement for 4/4 epochs.
[EARLY STOPPING] Triggered.
Stopped at epoch 14.

Training finished.
Best validation QWK: 0.6581
Best checkpoint path: /content/gpt_qwen_style_discourse_trait_stacks/best_gpt_qwen_style_discourse.pt


In [12]:
# =========================
# LOAD BEST CHECKPOINT AND EVALUATE ON TEST SET
# =========================

checkpoint = torch.load(
    best_path,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

print("Loaded checkpoint from:", best_path)
print("Best epoch:", checkpoint.get("epoch"))
print("Best validation QWK:", checkpoint.get("best_val_qwk"))
print("Selected metric:", checkpoint.get("selected_metric_name"))

test_metrics, y_true_band, y_pred_band, overall_pred = evaluate_model(
    model,
    test_loader,
    name="test best GPT + Qwen-style discourse",
)

# =========================
# EXPORT TEST PREDICTIONS
# =========================

def build_prediction_df(df, y_true_band, y_pred_band, overall_pred):
    out = df.reset_index(drop=True).copy()

    for i, trait in enumerate(TRAIT_COLS):
        out[f"gold_{trait}"] = y_true_band[:, i]
        out[f"pred_{trait}"] = y_pred_band[:, i]
        out[f"abs_error_{trait}"] = (
            out[f"pred_{trait}"] - out[f"gold_{trait}"]
        ).abs()

    out["pred_Overall_derived"] = overall_pred

    if "Overall" in out.columns:
        out["gold_Overall"] = out["Overall"]

        valid = ~pd.isna(out["gold_Overall"])

        if valid.any():
            print(
                "Derived Overall QWK:",
                qwk_band(
                    out.loc[valid, "gold_Overall"],
                    out.loc[valid, "pred_Overall_derived"],
                ),
            )

            print(
                "Derived Overall MAE:",
                mean_absolute_error(
                    out.loc[valid, "gold_Overall"],
                    out.loc[valid, "pred_Overall_derived"],
                ),
            )

    out["mean_abs_trait_error"] = out[
        [f"abs_error_{t}" for t in TRAIT_COLS]
    ].mean(axis=1)

    return out


pred_df = build_prediction_df(
    test_df,
    y_true_band,
    y_pred_band,
    overall_pred,
)

pred_path = os.path.join(
    OUTPUT_DIR,
    "test_predictions_gpt_qwen_style_discourse_trait_stacks.csv",
)

pred_df.to_csv(pred_path, index=False)

print("Saved:", pred_path)
display(pred_df.head())

# =========================
# EXPORT METRIC SUMMARY
# =========================

summary_rows = []

for trait in TRAIT_COLS:
    qwk = qwk_band(
        pred_df[f"gold_{trait}"],
        pred_df[f"pred_{trait}"],
    )

    mae = mean_absolute_error(
        pred_df[f"gold_{trait}"],
        pred_df[f"pred_{trait}"],
    )

    summary_rows.append({
        "model": "GPT + Qwen-style discourse",
        "target": trait,
        "QWK": qwk,
        "MAE": mae,
    })

if "gold_Overall" in pred_df.columns:
    valid = ~pd.isna(pred_df["gold_Overall"])

    if valid.any():
        qwk = qwk_band(
            pred_df.loc[valid, "gold_Overall"],
            pred_df.loc[valid, "pred_Overall_derived"],
        )

        mae = mean_absolute_error(
            pred_df.loc[valid, "gold_Overall"],
            pred_df.loc[valid, "pred_Overall_derived"],
        )

        summary_rows.append({
            "model": "GPT + Qwen-style discourse",
            "target": "Overall_derived",
            "QWK": qwk,
            "MAE": mae,
        })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(
    OUTPUT_DIR,
    "test_metric_summary_gpt_qwen_style_discourse.csv",
)

summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(summary_df)


Loaded checkpoint from: /content/gpt_qwen_style_discourse_trait_stacks/best_gpt_qwen_style_discourse.pt
Best epoch: 10
Best validation QWK: 0.6580501264920309
Selected metric: mean_trait_qwk


Predict:   0%|          | 0/31 [00:00<?, ?it/s]


== test best GPT + Qwen-style discourse ==
TR_qwk: 0.5829
TR_mae: 0.9268
CC_qwk: 0.5877
CC_mae: 1.0211
LR_qwk: 0.6347
LR_mae: 0.9387
GRA_qwk: 0.6681
GRA_mae: 0.9412
mean_trait_qwk: 0.6184
mean_trait_mae: 0.9570
derived_overall_from_traits_qwk: 0.6246
derived_overall_from_traits_mae: 0.9340
Saved: /content/gpt_qwen_style_discourse_trait_stacks/test_predictions_gpt_qwen_style_discourse_trait_stacks.csv


,prompt,essay,evaluation,essay_len,TR,CC,LR,GRA,n_found,parse_quality,...,pred_CC,abs_error_CC,gold_LR,pred_LR,abs_error_LR,gold_GRA,pred_GRA,abs_error_GRA,pred_Overall_derived,mean_abs_trait_error
0,18.Education of young people is highly priorit...,It can be argued that the vast majority of cou...,Task Achievement: \r\nThe essay adequately add...,303,7.0,7.0,6.0,6.0,4,good,...,7.5,0.5,6.0,7.0,1.0,6.0,7.0,1.0,7.0,0.750
1,Nowadays in many countries most shops and prod...,"In today's modern world, daily use things are ...",Task Achievement:\r\nThe candidate has adequat...,300,6.0,5.5,5.5,5.5,4,good,...,5.5,0.0,5.5,5.5,0.0,5.5,5.5,0.0,5.5,0.000
2,Some people believe that the rage of technolog...,Technology is deeply relative to the living of...,Task Achievement: 5.5\r\n- The candidate has e...,296,5.5,6.0,6.0,6.0,4,good,...,7.0,1.0,6.0,6.5,0.5,6.0,7.0,1.0,7.0,1.000
3,The world has many towns and cites constructed...,Numerous cities have houses and replanning and...,Task Achievement:\r\n\r\nThe essay adequately ...,342,6.5,6.5,6.0,6.0,4,good,...,7.0,0.5,6.0,6.5,0.5,6.0,6.5,0.5,7.0,0.500
4,Many people argue that in order to improve the...,"In this modern world, the importance of higher...",Task Achievement: (Band Score: 4)\r\n- The can...,264,4.0,3.5,3.0,3.0,4,good,...,5.5,2.0,3.0,5.0,2.0,3.0,5.0,2.0,5.0,1.875


Saved: /content/gpt_qwen_style_discourse_trait_stacks/test_metric_summary_gpt_qwen_style_discourse.csv


,model,target,QWK,MAE
0,GPT + Qwen-style discourse,TR,0.582902,0.926804
1,GPT + Qwen-style discourse,CC,0.587727,1.021134
2,GPT + Qwen-style discourse,LR,0.634733,0.938660
3,GPT + Qwen-style discourse,GRA,0.668074,0.941237
